# CASE expressions

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/sql-04-case-expressions.ipynb)

`CASE` is SQL's if/else. It turns values into labels (bucketing), into flags, and, inside an aggregate,
into columns (pivoting). Once you can write `CASE`, most reporting logic becomes a single query.

Previous: [NULL handling](sql-03-null-handling.ipynb). Next: [joins](sql-05-joins.ipynb).

## Setup

The first code cells install PySpark if it is missing, detect Java, and start a local Spark session.

In [ ]:
# Setup: installs this notebook's packages when they are missing (e.g. in Google Colab).
import importlib.util
import subprocess
import sys

PACKAGES = {"pyspark": "pyspark>=3.5,<4", "setuptools": "setuptools>=70"}


def _missing(module: str, spec: str) -> bool:
    """True when the module is absent or its installed version does not satisfy spec."""
    try:
        if importlib.util.find_spec(module) is None:
            return True
    except ModuleNotFoundError:
        return True
    try:
        from importlib.metadata import PackageNotFoundError, version

        from packaging.requirements import Requirement
    except ImportError:
        return False
    requirement = Requirement(spec)
    try:
        return not requirement.specifier.contains(version(requirement.name), prereleases=True)
    except PackageNotFoundError:
        return False


missing = [spec for module, spec in PACKAGES.items() if _missing(module, spec)]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
print("ready:", ", ".join(PACKAGES))

In [ ]:
# Detect PySpark and Java. Spark needs a Java runtime that can actually start a JVM.
import importlib.util
import shutil
import subprocess

HAS_PYSPARK = importlib.util.find_spec("pyspark") is not None
HAS_JAVA = shutil.which("java") is not None

# `shutil.which` only proves the binary exists; probe that it can start a JVM.
JAVA_RUNS = False
if HAS_JAVA:
    try:
        probe = subprocess.run(["java", "-version"], capture_output=True, text=True, timeout=60)
        JAVA_RUNS = probe.returncode == 0
    except Exception:
        JAVA_RUNS = False

SPARK_AVAILABLE = HAS_PYSPARK and HAS_JAVA and JAVA_RUNS

print(f"PySpark installed : {HAS_PYSPARK}")
print(f"Java on PATH      : {HAS_JAVA}")
print(f"Java runnable     : {JAVA_RUNS}")
print(f"Spark available   : {SPARK_AVAILABLE}")
if not SPARK_AVAILABLE:
    print("SKIPPED: Spark is not available, so every example prints SKIPPED instead of running.")

In [ ]:
import os
import sys
import time
from datetime import date, datetime
from decimal import Decimal

# Spark timestamps and Python datetimes must agree on the clock, so pin the process to UTC.
os.environ["TZ"] = "UTC"
if hasattr(time, "tzset"):
    time.tzset()

if SPARK_AVAILABLE:
    from pyspark.sql import SparkSession

    os.environ.setdefault("PYSPARK_PYTHON", sys.executable)
    spark = (
        SparkSession.builder.master("local[2]")
        .appName("sql-learning-path")
        .config("spark.sql.shuffle.partitions", "4")
        .config("spark.sql.session.timeZone", "UTC")
        .config("spark.ui.enabled", "false")
        .getOrCreate()
    )
    spark.sparkContext.setLogLevel("ERROR")
    print("Spark session ready:", spark.version)
else:
    spark = None


def create_view(name, ddl, rows):
    """Register rows as a temporary view. Skipped when Spark is unavailable."""
    if not SPARK_AVAILABLE:
        print(f"SKIPPED view {name}: needs a Spark session.")
        return
    spark.createDataFrame(rows, ddl).createOrReplaceTempView(name)
    print(f"view {name}: {spark.table(name).count()} rows")


def _plain(value):
    """Spark returns DECIMAL for literal arithmetic; compare those as floats."""
    return float(value) if isinstance(value, Decimal) else value


def check(query, expected, label):
    """Show a query's result, then assert it equals the expected rows. Skipped without Spark."""
    if not SPARK_AVAILABLE:
        print(f"SKIPPED {label}: needs a Spark session.")
        return
    frame = spark.sql(query)
    frame.show(truncate=False)
    rows = [tuple(_plain(value) for value in row) for row in frame.collect()]
    assert rows == expected, f"{label}: got {rows}, expected {expected}"
    print(f"PASS {label}")

## The example data

The same `customers` and `orders` tables as the earlier notebooks.

In [ ]:
create_view(
    "customers",
    "customer_id INT, name STRING, region STRING, signup_date DATE",
    [
        (1, "Acme Corp", "North", date(2022, 1, 5)),
        (2, "Beta LLC", "North", date(2022, 2, 10)),
        (3, "Cyan Inc", "South", date(2022, 3, 15)),
        (4, "Delta Co", "South", date(2022, 4, 20)),
        (5, "Echo Ltd", "West", date(2022, 5, 25)),
        (6, "Foxtrot GmbH", "West", date(2022, 6, 30)),
        (7, "Gamma SA", "East", date(2023, 1, 10)),
        (8, "Hotel BV", "East", date(2023, 2, 15)),
    ],
)

create_view(
    "orders",
    "order_id INT, customer_id INT, status STRING, amount INT, order_date DATE",
    [
        (1, 1, "completed", 100, date(2023, 1, 5)),
        (2, 1, "completed", 200, date(2023, 2, 10)),
        (3, 1, "completed", 150, date(2023, 3, 15)),
        (4, 2, "completed", 300, date(2023, 1, 20)),
        (5, 2, "cancelled", 250, date(2023, 2, 5)),
        (6, 2, "completed", 400, date(2023, 3, 20)),
        (7, 3, "completed", 120, date(2023, 1, 25)),
        (8, 3, "completed", 180, date(2023, 2, 28)),
        (9, 4, "pending", 220, date(2023, 2, 14)),
        (10, 4, "completed", 320, date(2023, 3, 30)),
        (11, 5, "completed", 500, date(2023, 1, 2)),
        (12, 5, "completed", 450, date(2023, 3, 5)),
        (13, 6, "completed", 275, date(2023, 2, 18)),
        (14, 6, "cancelled", 325, date(2023, 3, 25)),
    ],
)

## How `CASE` evaluates

`CASE` checks its `WHEN` branches from top to bottom and returns the value of the **first** branch that is
`TRUE`. The order of branches is therefore part of the logic. When no branch matches and there is no `ELSE`,
the result is `NULL`, not an error.

### Bucketing values

Label each completed order as `small`, `medium` or `large`. The order of the branches matters: `300` is checked before `150`.

In [ ]:
check(
    """
    SELECT order_id, amount,
           CASE WHEN amount >= 300 THEN 'large'
                WHEN amount >= 150 THEN 'medium'
                ELSE 'small'
           END AS size_bucket
    FROM orders
    WHERE status = 'completed'
    ORDER BY order_id
    """,
    [
        (1, 100, "small"), (2, 200, "medium"), (3, 150, "medium"), (4, 300, "large"),
        (6, 400, "large"), (7, 120, "small"), (8, 180, "medium"), (10, 320, "large"),
        (11, 500, "large"), (12, 450, "large"), (13, 275, "medium"),
    ],
    "completed orders bucketed by size",
)

### First match wins, and a missing `ELSE` is `NULL`

Both branches of the first example are true for 500, but only the first one is returned. The second
expression has no `ELSE` and no matching branch, so it is `NULL`.

In [ ]:
check(
    """
    SELECT
        CASE WHEN 500 >= 100 THEN 'a' WHEN 500 >= 300 THEN 'b' END AS first_match,
        CASE WHEN 1 = 2 THEN 'x' END AS no_else
    """,
    [("a", None)],
    "first match wins; missing ELSE is NULL",
)

### `CASE` and NULL

A `CASE` branch is `TRUE` only when its condition is `TRUE`. `WHEN value = NULL` is therefore never true, the
same trap as in `WHERE`. Test for missing values with `IS NULL`.

In [ ]:
check(
    """
    SELECT
        CASE WHEN CAST(NULL AS INT) = NULL THEN 'match' ELSE 'no match' END AS equality_test,
        CASE WHEN CAST(NULL AS INT) IS NULL THEN 'match' ELSE 'no match' END AS null_test
    """,
    [("no match", "match")],
    "equality with NULL never matches",
)

## Conditional aggregation: a pivot with `CASE`

Put a `CASE` inside an aggregate and each condition becomes its own column. `COUNT(CASE ... THEN 1 END)`
counts only the matching rows, because a `CASE` without `ELSE` yields `NULL`, and `COUNT` ignores `NULL`.
Customers with no orders keep a row and show zeros, because the join is a `LEFT JOIN`.

In [ ]:
check(
    """
    SELECT c.customer_id, c.name,
           COUNT(CASE WHEN o.status = 'completed' THEN 1 END) AS completed,
           COUNT(CASE WHEN o.status = 'pending'   THEN 1 END) AS pending,
           COUNT(CASE WHEN o.status = 'cancelled' THEN 1 END) AS cancelled
    FROM customers c
    LEFT JOIN orders o ON o.customer_id = c.customer_id
    GROUP BY c.customer_id, c.name
    ORDER BY c.customer_id
    """,
    [
        (1, "Acme Corp", 3, 0, 0), (2, "Beta LLC", 2, 0, 1), (3, "Cyan Inc", 2, 0, 0),
        (4, "Delta Co", 1, 1, 0), (5, "Echo Ltd", 2, 0, 0), (6, "Foxtrot GmbH", 1, 0, 1),
        (7, "Gamma SA", 0, 0, 0), (8, "Hotel BV", 0, 0, 0),
    ],
    "status counts as CASE columns",
)

## Spark's `PIVOT` clause

Spark SQL also has a `PIVOT` clause that generates the columns from the values in one column. It is shorter,
but it has two behaviours to know. Combinations that never occur become `NULL`, not zero. And a group that has
no matching rows at all, such as customers 7 and 8, disappears from the result. Wrap the pivot in `COALESCE`
when you need zeros.

In [ ]:
check(
    """
    SELECT customer_id,
           COALESCE(completed, 0) AS completed,
           COALESCE(pending, 0)   AS pending,
           COALESCE(cancelled, 0) AS cancelled
    FROM (
        SELECT * FROM (SELECT customer_id, status FROM orders)
        PIVOT (
            COUNT(*) FOR status IN (
                'completed' AS completed, 'pending' AS pending, 'cancelled' AS cancelled
            )
        )
    )
    ORDER BY customer_id
    """,
    [
        (1, 3, 0, 0), (2, 2, 0, 1), (3, 2, 0, 0),
        (4, 1, 1, 0), (5, 2, 0, 0), (6, 1, 0, 1),
    ],
    "PIVOT with zeros filled in",
)

## Takeaways

- A `CASE` returns the first matching branch, so order your conditions from most to least specific.
- Use `IS NULL` inside `CASE`; `= NULL` never matches.
- Conditional aggregation (`COUNT(CASE ...)`, `SUM(CASE ... ELSE 0 END)`) is the portable pivot. Use `PIVOT`
  when the column list is dynamic, and remember that `PIVOT` produces `NULL` for missing combinations.